In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.evaluate_models import eval_reg

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

print("Session 28 imports successful")

Session 28 imports successful


In [2]:
df = pd.read_csv(
    "../data/student-mat.csv",
    sep=";"
)

df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

X_full = df_encoded.drop(
    columns=["G3"]
).copy()

y = df_encoded["G3"].copy()

print("Dataset loaded.")
print("X_full shape:", X_full.shape)

Dataset loaded.
X_full shape: (395, 41)


In [3]:
Xtr_f, Xte_f, ytr, yte = train_test_split(
    X_full,
    y,
    test_size=0.20,
    random_state=42
)

print("Train/Test split complete.")
print("Training rows:", len(Xtr_f))
print("Test rows:", len(Xte_f))

Train/Test split complete.
Training rows: 316
Test rows: 79


In [4]:
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    Xtr_f,
    ytr
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [5]:
rf_predictions = rf.predict(
    Xte_f
)

print(
    "Predictions generated:",
    len(rf_predictions)
)

Predictions generated: 79


In [6]:
rf_results = eval_reg(
    yte,
    rf_predictions
)

rf_mae = rf_results["MAE"]
rf_rmse = rf_results["RMSE"]
rf_r2 = rf_results["R2"]

print("Random Forest test results:")
print(rf_results)

Random Forest test results:
{'MAE': 1.1706329113924052, 'RMSE': 1.97473322451841, 'R2': 0.8098238322966482}


In [7]:
rf_train_predictions = rf.predict(
    Xtr_f
)

rf_train_results = eval_reg(
    ytr,
    rf_train_predictions
)

print("Training Results")
print(rf_train_results)

print("\nTest Results")
print(rf_results)

Training Results
{'MAE': 0.33873417721518984, 'RMSE': 0.5610638879461441, 'R2': 0.9850130953359097}

Test Results
{'MAE': 1.1706329113924052, 'RMSE': 1.97473322451841, 'R2': 0.8098238322966482}


In [8]:
rf_train_rmse = rf_train_results["RMSE"]

rf_rmse_gap = (
    rf_rmse
    - rf_train_rmse
)

print(
    f"Train RMSE: {rf_train_rmse:.4f}"
)

print(
    f"Test RMSE: {rf_rmse:.4f}"
)

print(
    f"RMSE Gap: {rf_rmse_gap:.4f}"
)

Train RMSE: 0.5611
Test RMSE: 1.9747
RMSE Gap: 1.4137


In [9]:
tree_predictions = np.array([
    tree.predict(Xte_f)
    for tree in rf.estimators_
])

print(
    "Shape:",
    tree_predictions.shape
)

Shape: (300, 79)


/Users/jjun28/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/Users/jjun28/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/Users/jjun28/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/Users/jjun28/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warnings.warn(
/Users/jjun28/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but DecisionTreeRegressor was fitted without feature names
  warni

In [10]:
prediction_std = np.std(
    tree_predictions,
    axis=0
)

print(
    "Average prediction standard deviation:",
    prediction_std.mean()
)

print(
    "Maximum prediction standard deviation:",
    prediction_std.max()
)

Average prediction standard deviation: 1.4149228501809996
Maximum prediction standard deviation: 4.930732422492859


In [11]:
first_student_tree_predictions = (
    tree_predictions[:, 0]
)

manual_average = (
    first_student_tree_predictions.mean()
)

forest_prediction = (
    rf_predictions[0]
)

print(
    "Manual Average:",
    manual_average
)

print(
    "Forest Prediction:",
    forest_prediction
)

Manual Average: 8.27
Forest Prediction: 8.27


In [12]:
random_forest_row = pd.DataFrame([
    {
        "Model": "Random Forest",
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "R2": rf_r2
    }
])

random_forest_row.round(4)

,Model,MAE,RMSE,R2
0,Random Forest,1.1706,1.9747,0.8098


In [13]:
comparison_df = pd.DataFrame([
    {
        "Model": "KNN",
        "MAE": 2.5848101265822785,
        "RMSE": 3.3926950565642415,
        "R2": 0.4386562685587473
    },
    {
        "Model": "SVR",
        "MAE": 1.8366678536589514,
        "RMSE": 2.7260297218073055,
        "R2": 0.6375898115704413
    },
    {
        "Model": "Decision Tree",
        "MAE": 1.357218203737191,
        "RMSE": 2.467248497769575,
        "R2": 0.7031308891822728
    },
    {
        "Model": "Random Forest",
        "MAE": rf_mae,
        "RMSE": rf_rmse,
        "R2": rf_r2
    }
])

comparison_df = (
    comparison_df
    .sort_values(
        by="RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

comparison_df

,Model,MAE,RMSE,R2
0,Random Forest,1.170633,1.974733,0.809824
1,Decision Tree,1.357218,2.467248,0.703131
2,SVR,1.836668,2.726030,0.637590
3,KNN,2.584810,3.392695,0.438656
